# Entrega 1 — Limpieza e integración de datos ONSV

**Proyecto:** Factores recurrentes en siniestros de tránsito fatales en el Perú (2021–2025)  
**Integrantes:** Jordán Alvarado, Guillermo Sebastian; Marquez Assureira, Erick Andree; Munayco Meneses, Alessandra Sonaly  
**Curso:** Data Mining

Este notebook ordena el trabajo de la primera entrega: carga, diagnóstico, limpieza, validación de claves, integración y primeras visualizaciones. La unidad final será **un siniestro fatal**.

> Los archivos del ONSV son preliminares, por lo que los conteos podrían cambiar si la entidad actualiza la información.


## 1. Planteamiento del problema y unidad de análisis

El proyecto busca preparar una base que permita estudiar **patrones recurrentes en los siniestros de tránsito fatales** usando información del evento, los vehículos y las personas involucradas.

**Pregunta del proyecto:** ¿Qué características se repiten con mayor frecuencia en los siniestros fatales y qué tipologías podrían estudiarse posteriormente con técnicas de Data Mining?

**Objetivo de esta entrega:** limpiar e integrar las tres fuentes del ONSV para obtener una base consistente, con **1 fila = 1 siniestro fatal**, que pueda usarse después en clustering, PCA o reglas de asociación.

**Clave común:** `CODIGO_SINIESTRO`.


## 2. Librerías

In [1]:
# Si alguna librería no está instalada, se puede descomentar la siguiente línea:
# %pip install -q openpyxl plotly

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
import plotly.express as px
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)


## 3. Carga de las tres fuentes

Las bases provienen del **Observatorio Nacional de Seguridad Vial (ONSV)**. Los encabezados reales empiezan en la quinta fila de los Excel, por eso se utiliza `header=4`.

Fuente oficial: https://www.onsv.gob.pe/datosabiertos


In [2]:
NOMBRES = {
    "siniestros": "BBDD ONSV - SINIESTROS FATALES 2021-2025 (preliminar).xlsx",
    "vehiculos": "BBDD ONSV - VEHICULOS 2021-2025 (preliminar).xlsx",
    "personas": "BBDD ONSV - PERSONAS 2021-2025 (preliminar).xlsx",
}

URLS = {
    "siniestros": "https://www.onsv.gob.pe/estaticos/excel/BBDD%20ONSV%20-%20SINIESTROS%20FATALES%202021-2025%20%28preliminar%29.xlsx",
    "vehiculos": "https://www.onsv.gob.pe/estaticos/excel/BBDD%20ONSV%20-%20VEHICULOS%202021-2025%20%28preliminar%29.xlsx",
    "personas": "https://www.onsv.gob.pe/estaticos/excel/BBDD%20ONSV%20-%20PERSONAS%202021-2025%20%28preliminar%29.xlsx",
}

KEY = "CODIGO_SINIESTRO"

def buscar_archivo(nombre):
    cwd = Path.cwd()
    candidatos = [
        cwd / nombre,
        cwd / "data" / "raw" / nombre,
        cwd.parent / nombre,
        cwd.parent / "data" / "raw" / nombre,
    ]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    return None

def normalizar_columna(col):
    col = str(col).strip()
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = re.sub(r"[^0-9A-Za-z]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col.upper()

def leer_fuente(nombre_fuente):
    ruta = buscar_archivo(NOMBRES[nombre_fuente])
    origen = ruta if ruta is not None else URLS[nombre_fuente]
    df = pd.read_excel(origen, header=4)
    df.columns = [normalizar_columna(c) for c in df.columns]
    df = df.replace(r"^\s*$", np.nan, regex=True)
    return df

siniestros = leer_fuente("siniestros")
vehiculos = leer_fuente("vehiculos")
personas = leer_fuente("personas")

fuentes = {"Siniestros": siniestros, "Vehículos": vehiculos, "Personas": personas}

for nombre, df in fuentes.items():
    print(f"{nombre}: {df.shape[0]:,} filas x {df.shape[1]} columnas")


Siniestros: 9,106 filas x 27 columnas
Vehículos: 12,667 filas x 25 columnas
Personas: 25,412 filas x 32 columnas


## 4. Tabla resumen de las fuentes

Primero se revisa el tamaño, los tipos, los faltantes, la cantidad de valores distintos y ejemplos. Esto permite detectar problemas antes de modificar la información.


In [3]:
resumen_fuentes = pd.DataFrame([
    {
        "fuente": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
        "codigos_siniestro_unicos": df[KEY].nunique(),
        "duplicados_exactos": df.duplicated().sum(),
        "faltantes_promedio_pct": round(df.isna().mean().mean() * 100, 2),
    }
    for nombre, df in fuentes.items()
])

def resumen_variables(df, nombre):
    return pd.DataFrame({
        "fuente": nombre,
        "variable": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "n_faltantes": df.isna().sum().values,
        "porcentaje_faltantes": (df.isna().mean()*100).round(2).values,
        "valores_distintos": df.nunique(dropna=True).values,
        "ejemplos": [", ".join(df[c].dropna().astype(str).unique()[:3]) for c in df.columns],
    })

tabla_resumen_variables = pd.concat(
    [resumen_variables(df, nombre) for nombre, df in fuentes.items()],
    ignore_index=True
)

display(resumen_fuentes)
display(tabla_resumen_variables)


,fuente,filas,columnas,codigos_siniestro_unicos,duplicados_exactos,faltantes_promedio_pct
0,Siniestros,9106,27,9106,0,14.48
1,Vehículos,12667,25,9106,0,6.99
2,Personas,25412,32,9106,0,10.86


,fuente,variable,tipo,n_faltantes,porcentaje_faltantes,valores_distintos,ejemplos
0,Siniestros,CODIGO_SINIESTRO,str,0,0.00,9106,"A-2021-01-23, A-2021-01-248, A-2021-01-38"
1,Siniestros,FECHA_SINIESTRO,object,0,0.00,1729,"01/01/2021, 02/01/2021, 03/01/2021"
2,Siniestros,HORA_SINIESTRO,str,0,0.00,651,"04:40, 05:45, 06:00"
3,Siniestros,CLASE_SINIESTRO,str,0,0.00,11,"DESPISTE, ATROPELLO FUGA, CHOQUE"
4,Siniestros,CANTIDAD_DE_FALLECIDOS,int64,0,0.00,21,"1, 2, 5"
5,Siniestros,CANTIDAD_DE_LESIONADOS,int64,0,0.00,41,"0, 3, 1"
6,Siniestros,CANTIDAD_DE_VEHICULOS_DANADOS,float64,2813,30.89,9,"1.0, 0.0, 2.0"
7,Siniestros,DEPARTAMENTO,str,0,0.00,25,"LIMA, LA LIBERTAD, SAN MARTIN"
8,Siniestros,PROVINCIA,str,0,0.00,182,"HUARAL, LIMA, VIRU"
9,Siniestros,DISTRITO,str,0,0.00,1086,"HUARAL, PACHACAMAC, VIRU"


## 5. Diagnóstico de problemas

### 5.1 Valores faltantes

No se rellenan automáticamente. Primero se revisa su magnitud porque algunos campos pueden no aplicar o no haber sido registrados.


In [4]:
for nombre, df in fuentes.items():
    reporte = (df.isna().mean()*100).round(2).sort_values(ascending=False).head(10)
    print(f"\nTop de faltantes — {nombre}")
    display(reporte.rename("porcentaje_faltantes").to_frame())



Top de faltantes — Siniestros


,porcentaje_faltantes
CLASIFICACION_DE_LA_SENAL_VERTICAL_NO_2,94.14
CLASIFICACION_DE_LA_SENAL_VERTICAL_NO_1,85.55
EXISTE_SENAL_VERTICAL,77.86
EXISTE_SENAL_HORIZONTAL,77.86
CANTIDAD_DE_VEHICULOS_DANADOS,30.89
PERFIL_LONGITUDINAL_VIA,4.66
ZONIFICACION,4.65
CARACTERISTICAS_DE_VIA,4.65
CONDICION_CLIMATICA,4.65
SUPERFICIE_DE_CALZADA,4.65



Top de faltantes — Vehículos


,porcentaje_faltantes
TIPO_SEGURO,45.07
COMPANIA_SEGURO,37.74
AMBITO_SERVICIO,27.34
ELEMENTO_TRANSPORTADO,27.20
POSEE_SEGURO,16.97
POSEE_CITV,16.97
ESTADO_MODALIDAD,0.94
MODALIDAD_DE_TRANSPORTE,0.89
ESTADO_CITV,0.83
ESTADO_SOAT,0.77



Top de faltantes — Personas


,porcentaje_faltantes
RESULTADO_DEL_DOSAJE_ETILICO_CUALITATIVO,84.44
LUGAR_ATENCION_LESIONADO,72.17
POSEE_LICENCIA,71.76
LUGAR_DE_DEFUNCION,60.37
SE_SOMETIO_A_DOSAJE_ETILICO_CUANTITATIVO,16.85
SE_SOMETIO_A_DOSAJE_ETILICO_CUALITATIVO,15.31
PAIS_DE_NACIONALIDAD,13.02
EDAD,10.43
ESTADO_LICENCIA,1.15
SEXO,1.13


### 5.2 Duplicados y claves

Un `CODIGO_SINIESTRO` repetido en vehículos o personas **no es necesariamente un error**, porque un siniestro puede involucrar a varios vehículos y varias personas. Sí revisamos duplicados exactos y los identificadores específicos.


In [5]:
validacion_duplicados = pd.DataFrame({
    "fuente": ["Siniestros", "Vehículos", "Personas"],
    "filas": [len(siniestros), len(vehiculos), len(personas)],
    "duplicados_exactos": [siniestros.duplicated().sum(), vehiculos.duplicated().sum(), personas.duplicated().sum()],
    "duplicados_CODIGO_SINIESTRO": [
        siniestros.duplicated(subset=[KEY]).sum(),
        vehiculos.duplicated(subset=[KEY]).sum(),
        personas.duplicated(subset=[KEY]).sum(),
    ],
})
display(validacion_duplicados)

print("CODIGO_VEHICULO repetido:", vehiculos.duplicated(subset=["CODIGO_VEHICULO"]).sum())
print("CODIGO_PERSONA repetido:", personas.duplicated(subset=["CODIGO_PERSONA"]).sum())

display(
    personas[personas.duplicated(subset=["CODIGO_PERSONA"], keep=False)]
    [["CODIGO_SINIESTRO","CODIGO_VEHICULO","CODIGO_PERSONA","TIPO_PERSONA","GRAVEDAD","EDAD","SEXO"]]
)


,fuente,filas,duplicados_exactos,duplicados_CODIGO_SINIESTRO
0,Siniestros,9106,0,0
1,Vehículos,12667,0,3561
2,Personas,25412,0,16306


CODIGO_VEHICULO repetido: 0
CODIGO_PERSONA repetido: 1


,CODIGO_SINIESTRO,CODIGO_VEHICULO,CODIGO_PERSONA,TIPO_PERSONA,GRAVEDAD,EDAD,SEXO
14888,A-2023-04-103,V-2023-04-103-1,P-2023-04-103-1-2,PEATÓN,FALLECIDO,56,MASCULINO
14889,A-2023-04-103,V-2023-04-103-2,P-2023-04-103-1-2,PEATÓN,FALLECIDO,56,MASCULINO


### 5.3 Errores de registro, categorías y tipos

Se encontraron problemas simples de formato: espacios al final, diferencias entre mayúsculas y minúsculas, `NO INDICA` en edad y un símbolo `°` en una longitud. Estos casos se corrigen sin cambiar el significado de la información.


In [6]:
def diagnostico_texto(df, nombre):
    filas = []
    for col in df.select_dtypes(include=["object", "string"]).columns:
        s = df[col].dropna().astype(str)
        if s.empty:
            continue
        con_espacios = s.str.contains(r"^\s|\s$", regex=True).sum()
        n_raw = s.nunique()
        n_estandar = s.str.strip().str.upper().nunique()
        if con_espacios > 0 or n_estandar < n_raw:
            filas.append({
                "fuente": nombre,
                "variable": col,
                "filas_con_espacios": int(con_espacios),
                "categorias_antes": int(n_raw),
                "categorias_despues": int(n_estandar),
            })
    return pd.DataFrame(filas)

problemas_texto = pd.concat(
    [diagnostico_texto(df, nombre) for nombre, df in fuentes.items()],
    ignore_index=True
)
display(problemas_texto)

print("EDAD no convertible a número:")
edad_prueba = pd.to_numeric(personas["EDAD"], errors="coerce")
display(personas.loc[personas["EDAD"].notna() & edad_prueba.isna(), "EDAD"].value_counts())

print("Longitud no convertible directamente a número:")
lon_prueba = pd.to_numeric(siniestros["COORDENADAS_LONGITUD"], errors="coerce")
display(siniestros.loc[siniestros["COORDENADAS_LONGITUD"].notna() & lon_prueba.isna(), "COORDENADAS_LONGITUD"].value_counts())


,fuente,variable,filas_con_espacios,categorias_antes,categorias_despues
0,Siniestros,COD_CARRETERA,1,1045,1044
1,Siniestros,COORDENADAS_LONGITUD,2,9095,9095
2,Siniestros,PERFIL_LONGITUDINAL_VIA,6926,2,2
3,Vehículos,COMPANIA_SEGURO,2,26,26
4,Vehículos,CODIGO_DE_CARRETERA,1,1045,1044
5,Personas,CLASE_LICENCIA,0,23,22
6,Personas,CODIGO_DE_CARRETERA,2,1045,1044


EDAD no convertible a número:


EDAD
NO INDICA    31
Name: count, dtype: int64

Longitud no convertible directamente a número:


COORDENADAS_LONGITUD
-71.325300°    1
Name: count, dtype: int64

## 6. Limpieza y conversión de tipos

**Decisiones:**

- Se estandarizan textos con `strip()` y mayúsculas.
- Fechas pasan a `datetime`.
- Se crea `EDAD_NUMERICA`; `NO INDICA` queda como `NaN`.
- Se separan hora y minuto para poder validar rangos.
- Se retira `°` de la longitud antes de convertirla a número.
- No se hace imputación masiva de faltantes.
- No se eliminan filas completas porque no hay duplicados exactos y los valores extremos pueden ser eventos reales.


In [7]:
def estandarizar_texto(df):
    out = df.copy()
    for col in out.select_dtypes(include=["object", "string"]).columns:
        out[col] = out[col].astype("string").str.strip().str.upper()
        out[col] = out[col].replace("", pd.NA)
    return out

siniestros_limpio = estandarizar_texto(siniestros)
vehiculos_limpio = estandarizar_texto(vehiculos)
personas_limpio = estandarizar_texto(personas)

siniestros_limpio["FECHA_SINIESTRO"] = pd.to_datetime(siniestros_limpio["FECHA_SINIESTRO"], errors="coerce", dayfirst=True)
vehiculos_limpio["FECHA"] = pd.to_datetime(vehiculos_limpio["FECHA"], errors="coerce", dayfirst=True)
personas_limpio["FECHA"] = pd.to_datetime(personas_limpio["FECHA"], errors="coerce", dayfirst=True)

partes_hora = siniestros_limpio["HORA_SINIESTRO"].astype("string").str.extract(r"^\s*(\d{1,2}):(\d{2})\s*$")
siniestros_limpio["HORA_NUM"] = pd.to_numeric(partes_hora[0], errors="coerce")
siniestros_limpio["MINUTO_NUM"] = pd.to_numeric(partes_hora[1], errors="coerce")

personas_limpio["EDAD_NUMERICA"] = pd.to_numeric(personas_limpio["EDAD"], errors="coerce").astype("Int64")

siniestros_limpio["COORDENADAS_LONGITUD"] = pd.to_numeric(
    siniestros_limpio["COORDENADAS_LONGITUD"].astype("string").str.replace("°", "", regex=False).str.strip(),
    errors="coerce"
)
siniestros_limpio["COORDENADAS_LATITUD"] = pd.to_numeric(siniestros_limpio["COORDENADAS_LATITUD"], errors="coerce")

print("Limpieza y conversiones terminadas.")


Limpieza y conversiones terminadas.


### 6.1 Valores fuera de rango y posibles outliers

Se aplican reglas sencillas de plausibilidad. Los outliers se marcan para revisión, pero **no se eliminan automáticamente**.


In [8]:
reglas = []
def agregar_regla(nombre, mascara):
    reglas.append({"regla": nombre, "incumplimientos": int(mascara.fillna(False).sum())})

agregar_regla("Fallecidos >= 1", siniestros_limpio["CANTIDAD_DE_FALLECIDOS"] < 1)
agregar_regla("Lesionados >= 0", siniestros_limpio["CANTIDAD_DE_LESIONADOS"] < 0)
agregar_regla("Vehículos dañados >= 0", siniestros_limpio["CANTIDAD_DE_VEHICULOS_DANADOS"] < 0)
agregar_regla("Edad entre 0 y 120", personas_limpio["EDAD_NUMERICA"].notna() & ~personas_limpio["EDAD_NUMERICA"].between(0,120))
agregar_regla("Hora entre 0 y 23", siniestros_limpio["HORA_NUM"].notna() & ~siniestros_limpio["HORA_NUM"].between(0,23))
agregar_regla("Minuto entre 0 y 59", siniestros_limpio["MINUTO_NUM"].notna() & ~siniestros_limpio["MINUTO_NUM"].between(0,59))
agregar_regla("Latitud aproximada Perú (-19 a 0)", siniestros_limpio["COORDENADAS_LATITUD"].notna() & ~siniestros_limpio["COORDENADAS_LATITUD"].between(-19,0))
agregar_regla("Longitud aproximada Perú (-82.5 a -68)", siniestros_limpio["COORDENADAS_LONGITUD"].notna() & ~siniestros_limpio["COORDENADAS_LONGITUD"].between(-82.5,-68))

display(pd.DataFrame(reglas))

def resumen_outliers_iqr(df, columnas):
    filas=[]
    for col in columnas:
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        q1, q3 = s.quantile(.25), s.quantile(.75)
        iqr = q3-q1
        li, ls = q1-1.5*iqr, q3+1.5*iqr
        filas.append({
            "variable": col,
            "min": s.min(), "Q1": q1, "mediana": s.median(), "Q3": q3, "max": s.max(),
            "limite_inferior": li, "limite_superior": ls,
            "posibles_outliers": int(((s<li)|(s>ls)).sum())
        })
    return pd.DataFrame(filas)

display(resumen_outliers_iqr(
    siniestros_limpio,
    ["CANTIDAD_DE_FALLECIDOS","CANTIDAD_DE_LESIONADOS","CANTIDAD_DE_VEHICULOS_DANADOS"]
))


,regla,incumplimientos
0,Fallecidos >= 1,0
1,Lesionados >= 0,0
2,Vehículos dañados >= 0,0
3,Edad entre 0 y 120,0
4,Hora entre 0 y 23,0
5,Minuto entre 0 y 59,0
6,Latitud aproximada Perú (-19 a 0),0
7,Longitud aproximada Perú (-82.5 a -68),0


,variable,min,Q1,mediana,Q3,max,limite_inferior,limite_superior,posibles_outliers
0,CANTIDAD_DE_FALLECIDOS,1.0,1.0,1.0,1.0,33.0,1.0,1.0,929
1,CANTIDAD_DE_LESIONADOS,0.0,0.0,0.0,1.0,71.0,-1.5,2.5,779
2,CANTIDAD_DE_VEHICULOS_DANADOS,0.0,1.0,1.0,2.0,15.0,-0.5,3.5,25


## 7. Preparación para integrar

Como vehículos y personas tienen varias filas por siniestro, hacer un `merge` directo generaría duplicaciones. Por eso primero resumimos ambas tablas a **una fila por `CODIGO_SINIESTRO`**.

También se crean indicadores simples de disponibilidad (`1 = dato presente`, `0 = faltante`) para conservar información sobre la cobertura de edad, dosaje y seguro sin inventar valores.


In [9]:
personas_limpio["EDAD_DISPONIBLE"] = personas_limpio["EDAD_NUMERICA"].notna().astype(int)
personas_limpio["DOSAJE_CUALITATIVO_DISPONIBLE"] = personas_limpio["RESULTADO_DEL_DOSAJE_ETILICO_CUALITATIVO"].notna().astype(int)
vehiculos_limpio["TIPO_SEGURO_DISPONIBLE"] = vehiculos_limpio["TIPO_SEGURO"].notna().astype(int)
siniestros_limpio["SENAL_VERTICAL_2_DISPONIBLE"] = siniestros_limpio["CLASIFICACION_DE_LA_SENAL_VERTICAL_NO_2"].notna().astype(int)

def moda_segura(serie):
    serie = serie.dropna()
    if serie.empty:
        return pd.NA
    return serie.mode().iloc[0]

vehiculos_agg = vehiculos_limpio.groupby(KEY).agg(
    N_VEHICULOS=("CODIGO_VEHICULO", "nunique"),
    TIPOS_VEHICULO_DISTINTOS=("VEHICULO", "nunique"),
    VEHICULO_PRINCIPAL=("VEHICULO", moda_segura),
    MODALIDAD_PRINCIPAL=("MODALIDAD_DE_TRANSPORTE", moda_segura),
    N_CON_SEGURO=("POSEE_SEGURO", lambda s: (s == "SI").sum()),
    N_CON_CITV=("POSEE_CITV", lambda s: (s == "SI").sum()),
    PROP_TIPO_SEGURO_DISPONIBLE=("TIPO_SEGURO_DISPONIBLE", "mean"),
).reset_index()

personas_agg = personas_limpio.groupby(KEY).agg(
    N_PERSONAS=("CODIGO_PERSONA", "nunique"),
    EDAD_PROMEDIO=("EDAD_NUMERICA", "mean"),
    EDAD_MIN=("EDAD_NUMERICA", "min"),
    EDAD_MAX=("EDAD_NUMERICA", "max"),
    PROP_EDAD_DISPONIBLE=("EDAD_DISPONIBLE", "mean"),
    N_MASCULINO=("SEXO", lambda s: (s == "MASCULINO").sum()),
    N_FEMENINO=("SEXO", lambda s: (s == "FEMENINO").sum()),
    PROP_DOSAJE_CUALITATIVO_DISPONIBLE=("DOSAJE_CUALITATIVO_DISPONIBLE", "mean"),
).reset_index()

conteo_gravedad = pd.crosstab(personas_limpio[KEY], personas_limpio["GRAVEDAD"]).add_prefix("N_GRAVEDAD_").reset_index()
conteo_tipo_persona = pd.crosstab(personas_limpio[KEY], personas_limpio["TIPO_PERSONA"]).add_prefix("N_TIPO_PERSONA_").reset_index()

personas_agg = (
    personas_agg
    .merge(conteo_gravedad, on=KEY, how="left", validate="one_to_one")
    .merge(conteo_tipo_persona, on=KEY, how="left", validate="one_to_one")
)

print("Vehículos agregados:", vehiculos_agg.shape)
print("Personas agregadas:", personas_agg.shape)


Vehículos agregados: (9106, 8)
Personas agregadas: (9106, 19)


## 8. Validación de claves antes de integrar

En la tabla de siniestros y en las tablas agregadas la clave debe quedar única. En las tablas originales de vehículos y personas las repeticiones son esperadas por su unidad de análisis.


In [10]:
validacion_claves = pd.DataFrame({
    "fuente": ["Siniestros", "Vehículos originales", "Personas originales", "Vehículos agregados", "Personas agregadas"],
    "filas": [len(siniestros_limpio), len(vehiculos_limpio), len(personas_limpio), len(vehiculos_agg), len(personas_agg)],
    "codigos_unicos": [
        siniestros_limpio[KEY].nunique(), vehiculos_limpio[KEY].nunique(), personas_limpio[KEY].nunique(),
        vehiculos_agg[KEY].nunique(), personas_agg[KEY].nunique()
    ],
    "duplicados_clave": [
        siniestros_limpio.duplicated(subset=[KEY]).sum(),
        vehiculos_limpio.duplicated(subset=[KEY]).sum(),
        personas_limpio.duplicated(subset=[KEY]).sum(),
        vehiculos_agg.duplicated(subset=[KEY]).sum(),
        personas_agg.duplicated(subset=[KEY]).sum(),
    ]
})
display(validacion_claves)


,fuente,filas,codigos_unicos,duplicados_clave
0,Siniestros,9106,9106,0
1,Vehículos originales,12667,9106,3561
2,Personas originales,25412,9106,16306
3,Vehículos agregados,9106,9106,0
4,Personas agregadas,9106,9106,0


## 9. Integración auditada con `indicator=True`

Siguiendo el flujo de clase, usamos `how="outer"` e `indicator=True` para comprobar cuántos códigos aparecen en las dos fuentes y cuántos quedan sin cruce.

- `both`: aparece en ambas fuentes.
- `left_only`: solo aparece en siniestros.
- `right_only`: solo aparece en la otra fuente.


In [11]:
validacion_vehiculos = siniestros_limpio[[KEY]].merge(
    vehiculos_agg[[KEY]], on=KEY, how="outer", indicator=True, validate="one_to_one"
)
print("Siniestros + Vehículos")
display(validacion_vehiculos["_merge"].value_counts().reindex(["both","left_only","right_only"], fill_value=0).to_frame("cantidad"))

validacion_personas = siniestros_limpio[[KEY]].merge(
    personas_agg[[KEY]], on=KEY, how="outer", indicator=True, validate="one_to_one"
)
print("Siniestros + Personas")
display(validacion_personas["_merge"].value_counts().reindex(["both","left_only","right_only"], fill_value=0).to_frame("cantidad"))


Siniestros + Vehículos


,cantidad
_merge,
both,9106
left_only,0
right_only,0


Siniestros + Personas


,cantidad
_merge,
both,9106
left_only,0
right_only,0


**Interpretación de la validación:** los 9,106 códigos de siniestro encuentran correspondencia tanto en vehículos como en personas. En esta versión de las fuentes no quedan códigos `left_only` ni `right_only` después de la agregación.

## 10. Construcción de la base integrada para análisis

Después de auditar los cruces, se usa `left merge` tomando la tabla de siniestros como base. El resultado conserva **una fila por siniestro**.


In [12]:
base_integrada = (
    siniestros_limpio
    .merge(vehiculos_agg, on=KEY, how="left", validate="one_to_one")
    .merge(personas_agg, on=KEY, how="left", validate="one_to_one")
)

validacion_final = pd.DataFrame([
    {"control":"Filas de la base final", "resultado":len(base_integrada)},
    {"control":"Códigos de siniestro únicos", "resultado":base_integrada[KEY].nunique()},
    {"control":"Llaves faltantes", "resultado":int(base_integrada[KEY].isna().sum())},
    {"control":"Duplicados exactos", "resultado":int(base_integrada.duplicated().sum())},
    {"control":"Duplicados por CODIGO_SINIESTRO", "resultado":int(base_integrada.duplicated(subset=[KEY]).sum())},
])

print("Dimensión de la base integrada:", base_integrada.shape)
display(validacion_final)
display(base_integrada.head())


Dimensión de la base integrada: (9106, 55)


,control,resultado
0,Filas de la base final,9106
1,Códigos de siniestro únicos,9106
2,Llaves faltantes,0
3,Duplicados exactos,0
4,Duplicados por CODIGO_SINIESTRO,0


,CODIGO_SINIESTRO,FECHA_SINIESTRO,HORA_SINIESTRO,CLASE_SINIESTRO,CANTIDAD_DE_FALLECIDOS,CANTIDAD_DE_LESIONADOS,CANTIDAD_DE_VEHICULOS_DANADOS,DEPARTAMENTO,PROVINCIA,DISTRITO,ZONA,TIPO_DE_VIA,RED_VIAL,COD_CARRETERA,COORDENADAS_LATITUD,COORDENADAS_LONGITUD,CONDICION_CLIMATICA,ZONIFICACION,CARACTERISTICAS_DE_VIA,PERFIL_LONGITUDINAL_VIA,SUPERFICIE_DE_CALZADA,EXISTE_SENAL_VERTICAL,CLASIFICACION_DE_LA_SENAL_VERTICAL_NO_1,CLASIFICACION_DE_LA_SENAL_VERTICAL_NO_2,EXISTE_SENAL_HORIZONTAL,CAUSA_FACTOR_PRINCIPAL,CAUSA_ESPECIFICA,HORA_NUM,MINUTO_NUM,SENAL_VERTICAL_2_DISPONIBLE,N_VEHICULOS,TIPOS_VEHICULO_DISTINTOS,VEHICULO_PRINCIPAL,MODALIDAD_PRINCIPAL,N_CON_SEGURO,N_CON_CITV,PROP_TIPO_SEGURO_DISPONIBLE,N_PERSONAS,EDAD_PROMEDIO,EDAD_MIN,EDAD_MAX,PROP_EDAD_DISPONIBLE,N_MASCULINO,N_FEMENINO,PROP_DOSAJE_CUALITATIVO_DISPONIBLE,N_GRAVEDAD_FALLECIDO,N_GRAVEDAD_ILESO,N_GRAVEDAD_LESIONADO,N_GRAVEDAD_NO SE CONOCE,N_TIPO_PERSONA_CONDUCTOR,N_TIPO_PERSONA_CONDUCTOR FUGADO,N_TIPO_PERSONA_OCUPANTE,N_TIPO_PERSONA_PASAJERO,N_TIPO_PERSONA_PEATÓN,N_TIPO_PERSONA_SIN CONDUCTOR
0,A-2021-01-23,2021-01-01,04:40,DESPISTE,1,0,1.0,LIMA,HUARAL,HUARAL,RURAL,CARRETERA,PROVINCIAL,LM-671,-11.482879,-77.255547,DESPEJADO,RURAL,TRAMO RECTO,PLANA,TROCHA,<NA>,<NA>,<NA>,<NA>,EN PROCESO DE INVESTIGACIÓN,NO CUENTA CON CAUSA ESPECIFICA,4,40,0,1,1,CAMIONETA RURAL,PARTICULAR,0,0,0.0,1,21.0,21,21,1.000000,1,0,0.0,1,0,0,0,1,0,0,0,0,0
1,A-2021-01-248,2021-01-01,05:45,DESPISTE,1,3,0.0,LIMA,LIMA,PACHACAMAC,URBANA,AVENIDA,URBANO,NO CORRESPONDE,-12.229400,-76.859412,DESPEJADO,INDUSTRIAL,TRAMO RECTO,PLANA,ASFALTADA,<NA>,<NA>,<NA>,<NA>,IMPRUDENCIA DEL CONDUCTOR,CONDUCIR EN ESTADO DE EBRIEDAD Y/O DROGADICCIÓN,5,45,0,1,1,TRIMOTO PASAJERO,SERVICIO ESPECIAL DE PASAJEROS,0,0,0.0,4,29.25,0,64,1.000000,3,1,0.5,1,0,3,0,1,0,0,3,0,0
2,A-2021-01-38,2021-01-01,06:00,ATROPELLO FUGA,2,0,NaN,LA LIBERTAD,VIRU,VIRU,URBANA,CARRETERA,PROVINCIAL,LI-1150,-8.414865,-78.754544,DESPEJADO,COMERCIAL,TRAMO RECTO,PLANA,ASFALTADA,<NA>,<NA>,<NA>,<NA>,EN PROCESO DE INVESTIGACIÓN,NO CUENTA CON CAUSA ESPECIFICA,6,0,0,1,1,VEHICULO NO IDENTIFICADO,<NA>,0,0,0.0,2,38.0,38,38,0.500000,1,0,0.0,2,0,0,0,0,1,0,0,1,0
3,A-2021-01-39,2021-01-01,07:00,CHOQUE,1,0,2.0,LA LIBERTAD,VIRU,VIRU,RURAL,CARRETERA,NACIONAL,PE-1N,-8.432617,-78.772242,DESPEJADO,RURAL,TRAMO RECTO,PLANA,ASFALTADA,<NA>,<NA>,<NA>,<NA>,IMPRUDENCIA DEL CONDUCTOR,GIRAR IMPRUDENTEMENTE,7,0,0,2,1,MOTOCICLETA,PARTICULAR,0,0,0.5,3,41.0,34,48,0.666667,2,0,0.0,1,0,0,2,0,2,0,1,0,0
4,A-2021-01-254,2021-01-01,14:00,ATROPELLO,1,1,0.0,LIMA,LIMA,VILLA MARIA DEL TRIUNFO,URBANA,AVENIDA,URBANO,NO CORRESPONDE,-12.164406,-76.953426,DESPEJADO,INDUSTRIAL,TRAMO RECTO,PLANA,ASFALTADA,<NA>,<NA>,<NA>,<NA>,IMPRUDENCIA DEL CONDUCTOR,CONDUCIR EN ESTADO DE EBRIEDAD Y/O DROGADICCIÓN,14,0,0,1,1,MOTOCICLETA,PARTICULAR,0,0,1.0,2,23.0,16,30,1.000000,2,0,0.5,1,0,1,0,1,0,0,0,1,0


## 11. Tres gráficos iniciales con Plotly

Estas visualizaciones son exploratorias. Sirven para presentar primeras lecturas de la base, pero todavía no corresponden al modelamiento de Data Mining.


### 11.1 Top 10 departamentos con más siniestros fatales registrados

In [13]:
top_departamentos = (
    base_integrada["DEPARTAMENTO"]
    .fillna("NO REGISTRA")
    .value_counts()
    .head(10)
    .sort_values()
    .reset_index()
)
top_departamentos.columns = ["departamento", "cantidad"]

fig = px.bar(
    top_departamentos,
    x="cantidad", y="departamento",
    orientation="h",
    text="cantidad",
    title="Top 10 departamentos por siniestros fatales registrados",
    labels={"cantidad":"Número de siniestros", "departamento":"Departamento"}
)
fig.show()


**Interpretación:** Lima registra la mayor cantidad de siniestros fatales en la base, con **1,917 casos (21.1%)**. Después aparecen La Libertad y Cusco. Este gráfico muestra conteos absolutos; por eso no permite afirmar que Lima tenga necesariamente un mayor riesgo, ya que no se está ajustando por población, parque vehicular o exposición al tránsito.

### 11.2 Cantidad de siniestros según clase

In [14]:
conteo_clase = (
    base_integrada["CLASE_SINIESTRO"]
    .fillna("NO REGISTRA")
    .value_counts()
    .sort_values()
    .reset_index()
)
conteo_clase.columns = ["clase_siniestro", "cantidad"]

fig = px.bar(
    conteo_clase,
    x="cantidad", y="clase_siniestro",
    orientation="h",
    text="cantidad",
    title="Cantidad de siniestros fatales según clase de siniestro",
    labels={"cantidad":"Número de siniestros", "clase_siniestro":"Clase de siniestro"}
)
fig.show()


**Interpretación:** el **choque** es la clase más frecuente (2,875 casos), seguido por el **despiste** (2,571) y el **atropello** (1,871). En conjunto, estas tres categorías representan aproximadamente **80.4%** de los siniestros registrados, por lo que son las clases con mayor presencia en la base.

### 11.3 Vehículos y personas involucradas por siniestro

In [15]:
base_grafico = base_integrada.copy()
base_grafico["ZONA_GRAFICO"] = base_grafico["ZONA"].fillna("NO REGISTRA")

fig = px.scatter(
    base_grafico,
    x="N_VEHICULOS",
    y="N_PERSONAS",
    color="ZONA_GRAFICO",
    size="CANTIDAD_DE_FALLECIDOS",
    hover_data=[KEY, "CLASE_SINIESTRO"],
    title="Vehículos y personas involucradas por siniestro",
    labels={
        "N_VEHICULOS":"Número de vehículos",
        "N_PERSONAS":"Número de personas",
        "ZONA_GRAFICO":"Zona",
        "CANTIDAD_DE_FALLECIDOS":"Fallecidos"
    }
)
fig.show()


**Interpretación:** la mayor parte de los puntos se concentra en pocos vehículos y pocas personas. La mediana es de **1 vehículo** y **2 personas** por siniestro. También aparecen algunos casos mucho más grandes, con hasta **74 personas** y **5 vehículos**. Este gráfico es importante porque combina variables que originalmente estaban separadas en las tres fuentes.

## 12. Tres hallazgos preliminares

In [16]:
clases_validas = base_integrada["CLASE_SINIESTRO"].dropna()
clase_principal = clases_validas.value_counts().idxmax()
n_clase_principal = int(clases_validas.value_counts().max())

hallazgos = pd.DataFrame({
    "Hallazgo": [1,2,3],
    "Interpretación": [
        f"Lima concentra {int((base_integrada['DEPARTAMENTO']=='LIMA').sum()):,} registros. Es un conteo absoluto y no una tasa de riesgo.",
        f"La clase más frecuente es {clase_principal}, con {n_clase_principal:,} casos; choque, despiste y atropello concentran la mayoría de registros.",
        f"La mediana es de {base_integrada['N_VEHICULOS'].median():.0f} vehículo y {base_integrada['N_PERSONAS'].median():.0f} personas por siniestro, aunque existen eventos de mayor tamaño."
    ]
})
display(hallazgos)


,Hallazgo,Interpretación
0,1,"Lima concentra 1,917 registros. Es un conteo a..."
1,2,"La clase más frecuente es CHOQUE, con 2,875 ca..."
2,3,La mediana es de 1 vehículo y 2 personas por s...


## 13. Exportación y siguiente paso

La base queda preparada para la siguiente fase del proyecto. En esta Entrega 1 **todavía no se aplica One-Hot Encoding, escalamiento ni clustering**; esas transformaciones se harán cuando se definan las variables que entrarán al modelo.


In [17]:
base_integrada.to_csv("base_integrada_onsv_entrega1.csv", index=False, encoding="utf-8-sig")
print("Archivo generado: base_integrada_onsv_entrega1.csv")


Archivo generado: base_integrada_onsv_entrega1.csv
